# Offline evaluation on `shop.history_sessions`

Before putting a new recommender live, estimate offline whether it beats the baseline.

The log was produced by the **old system** (best sellers plus a bit of randomness), so it only tells you what happened with the products *it* showed. Two estimates:

1. **Hit rate**: of the impressions where the shopper clicked, how often is the clicked product in your top 10? Easy, but biased towards popular products (the old system showed them most).
2. **Inverse propensity scoring (IPS)** of the click rate in **slot 1**: keep only the impressions where your first product equals the one the old system put in slot 1, and weight each by `1 / propensity`. This corrects the popularity bias of the log for that slot. SNIPS (self-normalised IPS) has lower variance.

We hold out the **last 7 days** of the history: the policies are built only from data before the cut and evaluated after it.

In [95]:
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests

for line in Path("../.env").read_text().splitlines() if Path("../.env").exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())
API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
http = requests.Session()
http.headers["Authorization"] = f"Bearer {os.environ['RAWTREE_API_KEY']}"

def sql(q):
    r = http.post(f"{API}/v1/query", json={"sql": q}, timeout=600)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

## Data

In [96]:
last_day = sql("SELECT toDate(max(ts::DateTime64(3))) AS d FROM shop.history_sessions").d[0]
CUT = str((pd.Timestamp(last_day) - pd.Timedelta(days=7)).date())   # 'YYYY-MM-DD'
print("train < ", CUT, "<= test")

# Columns are Dynamic (schemaless): filters work as is; joins and some aggregates need casts like ::String
test = sql(f"""
    SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
    FROM shop.history_sessions WHERE ts >= '{CUT}' ORDER BY rand() LIMIT 100000""")
test["clicked"] = test.clicked.replace("", None)
print(len(test), "test impressions, observed CTR", round(test.clicked.notna().mean(), 4))
test.head(3)

train <  2026-11-07 <= test
100000 test impressions, observed CTR 0.0369


,request_id,ts,user_id,context,recent_views,shown,propensity,clicked,purchased
0,r_5f6a9fd775d7423b,2026-11-12 00:45:54.694000000,u_2004988d,"{'device': 'web', 'page': 'product', 'region':...","[{'product_id': 'p_819e00', 'ts': '2026-11-12 ...","[p_69d40e, p_8a86a8, p_0ef052, p_c810ca, p_ada...","[0.451662, 0.10555, 0.002287, 0.015664, 0.0876...",NaN,NaN
1,r_705ba965ff0308e6,2026-11-10 22:22:08.368000000,u_51709e4d,"{'device': 'android', 'page': 'category', 'reg...","[{'product_id': 'p_79b179', 'ts': '2026-11-10 ...","[p_86e4e5, p_6d0f23, p_dbd2ba, p_1516d4, p_1bc...","[0.004996, 0.021569, 0.019285, 0.001976, 0.176...",NaN,NaN
2,r_ea53404bc1c9fa3a,2026-11-07 14:52:16.601000000,u_083d091c,"{'device': 'android', 'page': 'product', 'regi...","[{'product_id': 'p_9632e5', 'ts': '2026-11-07 ...","[p_13be1a, p_c98f7f, p_4d2261, p_2c0c81, p_3d9...","[0.111716, 0.035069, 0.008264, 0.015412, 0.255...",NaN,NaN


In [97]:
import sys
sys.path.insert(0, ".")
from recommender import best_sellers, recommend   # the same code that runs live

best = best_sellers(until=CUT)

In [98]:
# Co-views computed inside RawTree: for every product, the 20 products most viewed in the same sessions
coviews = sql(f"""
    SELECT a.product_id AS p, b.product_id AS q, count() AS n
    FROM (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS a
    INNER JOIN (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS b
      ON a.session_id = b.session_id
    WHERE a.product_id != b.product_id
    GROUP BY p, q ORDER BY n DESC LIMIT 20 BY p""")
covisit = coviews.groupby("p").q.apply(list).to_dict()
print(len(covisit), "products with co-views")

5000 products with co-views


## Policies

A policy maps one request (a row) to an ordered list of 10 product IDs.

In [99]:
def baseline(row):
    """The live baseline, imported from starter/recommender.py."""
    return recommend(row.recent_views, best)

def coview(row):
    """Products co-viewed with this session's views (most recent first), then the baseline."""
    score = Counter()
    for k, v in enumerate(reversed(row.recent_views)):
        for rank, q in enumerate(covisit.get(v["product_id"], [])):
            score[q] += 0.7 ** k / (rank + 1)
    items = [q for q, _ in score.most_common(10)]
    return (items + [p for p in baseline(row) if p not in items])[:10]

## Evaluation

In [100]:
POLICIES = {"baseline": baseline, "coview": coview}

def evaluate(policy, df):
    recs = [policy(r) for r in df.itertuples()]
    clicked = df.clicked.to_numpy()
    has_click = df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has_click].mean()
    first_shown = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first_shown)])
    reward = (clicked == first_shown).astype(float)            # click on the slot-1 product
    w = match / df.propensity.str[0].to_numpy()
    return {"hit_rate@10": hit, "IPS_ctr_slot1": (w * reward).mean(),
            "SNIPS_ctr_slot1": (w * reward).sum() / max(w.sum(), 1e-9), "slot1_matches": int(match.sum())}

old_slot1 = (test.clicked == test.shown.str[0]).mean()
print(f"old system, observed slot-1 CTR: {old_slot1:.4f}")
pd.DataFrame({name: evaluate(p, test) for name, p in POLICIES.items()}).T

old system, observed slot-1 CTR: 0.0109


,hit_rate@10,IPS_ctr_slot1,SNIPS_ctr_slot1,slot1_matches
baseline,0.390290,0.006030,0.041677,3887.0
coview,0.230811,0.007845,0.011374,5527.0


## Next steps

- Few `slot1_matches` means a noisy IPS estimate: report it next to the estimate.
- Personal taste: who bought what in `shop.history_events` (matrix factorisation, item-to-item similarity).
- Cold start: new products only have a `description`. Text embeddings can place them next to similar products.
- Context: does price sensitivity change with `device` and `region`?
- Live: the stream changes (trends, price drops, stock-outs). Your `team_NN.feedback` tells you how you are really doing.

___

Métricas:
- hit_rate@10 = (# clicked products in top 10) / (# observed clicks)
- IPC_ctr_slot1
- SNIPS_ctr_slot1
- slot1_matches: # observations where the new policy's first recommendation matches the logging policy's first recommendation

### Two tower

Idea: entrenar una NN con dos partes fundamentales:
1. Embeddings the los productos
2. Embeddings de los usuarios

Salida: producto vectorial de ambos embeddings = score de este producto para este usuario.

Los embeddings de los productos se precomputan en cuanto se puede.

Los embeddings de los usuarios tienen en cuenta los productos.
- Básicamente los fusionamos en un solo vector (por ejemplo con la media) y alimentamos al MLP junto con los datos del usuario.

- Cold start: podemos intentar usar los productos top populares para usuarios sin historial. Podemos, incluso, dar algunas compras de margen antes de empezar usar su historial.

El problema es el timpo. Como mínimo, habrá que seleccionar las últimas N compras. M
as sofisticado: podemos usar algo como un exponential time decay.

Importante: hay que recordar que no se puede sugerir un producto ya comprado (¿o recientemente comprado?).

Posible approach:

#### Redes recurrentes

Son un posible approach. En resumen: tenemos una red recurrente con pesos compartidos.

- En texto es fácil de entender: "the cat sat on...."
- En productos, es similar: "zapatos", "chaqueta", ...

Para distinguir a un usuario de otro, podemos guardar unos estados para cada usuario. Otra opción sería concatenar el embedding del usuario en cada paso.

___

When both towers receive an input, this input is linked. It is linked through one interaction, a.k.a. request. A request basically says "this user was in this context at this timestamp". The user is basically getting to our page, and we must display 10 products.

This is like a row or our training data, but both towers will receive different things.

See, the product tower will receive one product, of course, and its data. I already like the implementation. Once we have this one, we will be able to precompute the embeddings of the products.

The user tower is going to receive a user and some of it's last interactions. Let's say the last 50 interactions. This tower is more dynamic, we can't precompute it (not completely). We must take the timestamp of the request to take the last 50 interactions of that user. I will take care of how to compute one interaction. This user, the user that is making the request, has:
- A device (3 classes)
- A region (around 10 classes, we'll do 16 in case)
- A page (3 classes)

And all of those 50 interactions (a.k.a recent_views) will have:
- A timestamp (we will use it if we want to give greater value to most recent views)
- A product id

Of course, this dynamic data will be averaged (preferably taking into acount the timestamp, so older affects less) resulting in a single embedding, the same size of the products embedding, computed using the opposite tower and then averaged.

In [101]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [102]:
HIDDEN_1_DIM = 256
EMBEDDING_DIM = 128
MAX_HISTORY = 500

#### ProductTower

In [103]:
class ProductTower(nn.Module):

    def __init__(
        self,

        num_categories=11,
        num_subcategories=91,
        num_brands=129,

        category_embedding_dim=16,
        subcategory_embedding_dim=32,
        brand_embedding_dim=32,

        hidden_dim=HIDDEN_1_DIM,
        output_dim=EMBEDDING_DIM,
    ):
        super().__init__()

        self.category_embedding = nn.Embedding(
            num_categories,
            category_embedding_dim,
        )

        self.subcategory_embedding = nn.Embedding(
            num_subcategories,
            subcategory_embedding_dim,
        )

        self.brand_embedding = nn.Embedding(
            num_brands,
            brand_embedding_dim,
        )

        input_dim = (
            category_embedding_dim
            + subcategory_embedding_dim
            + brand_embedding_dim
            + 1
        )

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(
        self,
        category,
        subcategory,
        brand,
        price,
    ):

        # Categorical embeddings
        category_emb = self.category_embedding(category)
        subcategory_emb = self.subcategory_embedding(subcategory)
        brand_emb = self.brand_embedding(brand)

        # Numerical feature
        price = torch.log1p(price.clamp(min=0)).unsqueeze(-1)

        # Concatenate along the feature dimension.
        # Supports both [B] and [B, H] inputs.

        x = torch.cat(
            [
                category_emb,
                subcategory_emb,
                brand_emb,
                price,
            ],
            dim=-1,
        )

        # Neural network
        embedding = self.network(x)

        # Normalize along the embedding dimension.
        embedding = F.normalize(
            embedding,
            p=2,
            dim=-1,
        )

        return embedding


In [104]:
# TODO: una vez entrenado todo el sistema
    # Intentar comparar objetos que sean claramente similares y objetos claramente distintos

# similarity = F.cosine_similarity(
#     embedding_product_a,
#     embedding_product_b,
#     dim=1
# )

#### UserTower

In [105]:
class UserTower(nn.Module):

    def __init__(
        self,
        num_devices=4,
        num_regions=16,
        num_pages=4,

        product_embedding_dim=EMBEDDING_DIM,

        device_embedding_dim=4,
        region_embedding_dim=8,
        page_embedding_dim=4,

        hidden_dim=256,
        output_dim=EMBEDDING_DIM,

        recency_tau=24.0,
    ):
        super().__init__()

        self.device_embedding = nn.Embedding(
            num_devices, device_embedding_dim
        )

        self.region_embedding = nn.Embedding(
            num_regions, region_embedding_dim
        )

        self.page_embedding = nn.Embedding(
            num_pages, page_embedding_dim
        )

        self.recency_tau = recency_tau

        context_dim = (
            device_embedding_dim
            + region_embedding_dim
            + page_embedding_dim
        )

        input_dim = product_embedding_dim + context_dim

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(
        self,
        device,
        region,
        page,

        recent_product_embeddings,
        recent_timestamps,
        history_mask,

        request_timestamp,
    ):

        # [B, H] interaction age in hours

        age_hours = (
            request_timestamp.unsqueeze(1)
            - recent_timestamps
        ) / 3600.0

        age_hours = age_hours.clamp(min=0)

        weights = torch.exp(
            -math.log(2)
            * age_hours
            / self.recency_tau
        )

        weights = weights * history_mask.float()

        weights = weights.unsqueeze(-1)

        weighted_history = (
            recent_product_embeddings * weights
        ).sum(dim=1)

        weight_sum = weights.sum(dim=1)

        history_embedding = weighted_history / weight_sum.clamp(min=1e-8)

        # Context embeddings

        device_emb = self.device_embedding(device)
        region_emb = self.region_embedding(region)
        page_emb = self.page_embedding(page)

        context = torch.cat(
            [device_emb, region_emb, page_emb],
            dim=-1,
        )

        # Combine history and context

        x = torch.cat(
            [history_embedding, context],
            dim=-1,
        )

        embedding = self.network(x)

        return F.normalize(
            embedding,
            p=2,
            dim=-1,
        )


In [106]:
class TwoTowers(nn.Module):

    def __init__(
        self,
        product_tower,
        user_tower,
    ):
        super().__init__()

        self.product_tower = product_tower
        self.user_tower = user_tower

        # Learnable inverse temperature.
        self.logit_scale = nn.Parameter(
            torch.tensor(math.log(1 / 0.07))
        )

    def forward(
        self,
        device,
        region,
        page,

        history_category,
        history_subcategory,
        history_brand,
        history_price,
        history_timestamps,
        history_mask,

        request_timestamp,

        candidate_category,
        candidate_subcategory,
        candidate_brand,
        candidate_price,
    ):

        batch_size, num_candidates = candidate_category.shape

        # ---------------------------------
        # 1. Encode historical products
        # ---------------------------------

        history_embeddings = self.product_tower(
            history_category,
            history_subcategory,
            history_brand,
            history_price,
        )

        # [B, H, D]

        # ---------------------------------
        # 2. Encode user context
        # ---------------------------------

        user_embedding = self.user_tower(
            device,
            region,
            page,

            history_embeddings,
            history_timestamps,
            history_mask,

            request_timestamp,
        )

        # [B, D]

        # ---------------------------------
        # 3. Encode candidate products
        # ---------------------------------

        # Flatten candidate dimensions.

        category = candidate_category.reshape(-1)
        subcategory = candidate_subcategory.reshape(-1)
        brand = candidate_brand.reshape(-1)
        price = candidate_price.reshape(-1)

        candidate_embeddings = self.product_tower(
            category,
            subcategory,
            brand,
            price,
        )

        # [B * C, D]

        candidate_embeddings = candidate_embeddings.view(
            batch_size,
            num_candidates,
            -1,
        )

        # ---------------------------------
        # 4. Calculate similarity
        # ---------------------------------

        # [B, 1, D] x [B, C, D]

        similarity = torch.sum(
            user_embedding.unsqueeze(1)
            * candidate_embeddings,
            dim=-1,
        )

        # [B, C]

        scale = self.logit_scale.exp().clamp(max=100)

        logits = scale * similarity

        return logits


Es muy importante que siempre usemos los mismos datos de entrenamiento.

Cogemos siempre datasets anteriores a CUT.

In [116]:
def download_training_data() -> list[dict]:

    train = sql(f"""
        SELECT
            request_id,
            ts,
            user_id,
            context,
            recent_views,
            shown,
            propensity,
            clicked,
            purchased
        FROM shop.history_sessions
        WHERE ts <= '{CUT}'
        ORDER BY ts
        LIMIT 100000
    """)

    train["clicked"] = train["clicked"].replace("", None)
    train["purchased"] = train["purchased"].replace("", None)

    examples = []

    for row in train.itertuples():

        shown = row.shown

        if shown is None or (
            isinstance(shown, float) and pd.isna(shown)
        ):
            shown = []

        shown = list(dict.fromkeys(
            str(pid)
            for pid in shown
            if pid is not None
        ))

        if not shown:
            continue

        clicked = row.clicked
        purchased = row.purchased

        positives = set()

        if clicked is not None and not pd.isna(clicked):
            positives.add(str(clicked))

        if purchased is not None and not pd.isna(purchased):
            positives.add(str(purchased))

        # Only products that were actually shown
        # can be positive labels.

        positives &= set(shown)

        if not positives:
            continue

        context = row.context
        recent_views = row.recent_views

        examples.append({
            "request_id": row.request_id,
            "ts": row.ts,
            "user_id": row.user_id,
            "context": context if isinstance(context, dict) else {},
            "recent_views": recent_views if isinstance(recent_views, list) else [],
            "shown": shown,
            "positives": list(positives),
            "propensity": row.propensity,
        })

    return examples

In [117]:
def timestamp_to_unix(value):

    if value is None or pd.isna(value):
        return 0.0

    ts = pd.Timestamp(value)

    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")
    else:
        ts = ts.tz_convert("UTC")

    return float(ts.timestamp())


def encode_category(value, mapping):
    if value is None:
        return 0

    return mapping.get(str(value), 0)


class ImpressionDataset(Dataset):

    def __init__(
        self,
        examples,
        product_features,
        context_encoders,
        max_history=MAX_HISTORY,
    ):
        self.examples = examples
        self.product_features = product_features
        self.context_encoders = context_encoders
        self.max_history = max_history

    def encode_product(self, product_id):

        return self.product_features.get(
            str(product_id),
            {
                "category": 0,
                "subcategory": 0,
                "brand": 0,
                "price": 0.0,
            },
        )

    def encode_products(self, product_ids):

        features = [
            self.encode_product(pid)
            for pid in product_ids
        ]

        return {
            "category": torch.tensor(
                [x["category"] for x in features],
                dtype=torch.long,
            ),
            "subcategory": torch.tensor(
                [x["subcategory"] for x in features],
                dtype=torch.long,
            ),
            "brand": torch.tensor(
                [x["brand"] for x in features],
                dtype=torch.long,
            ),
            "price": torch.tensor(
                [x["price"] for x in features],
                dtype=torch.float32,
            ),
        }

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):

        row = self.examples[idx]

        context = row["context"]

        # -------------------------
        # 1. Context
        # -------------------------

        device = encode_value(
            context.get("device"),
            self.context_encoders["device"],
        )

        region = encode_value(
            context.get("region"),
            self.context_encoders["region"],
        )

        page = encode_value(
            context.get("page"),
            self.context_encoders["page"],
        )

        # -------------------------
        # 2. History
        # -------------------------

        recent_views = row["recent_views"]

        # Filter malformed events first.

        recent_views = [
            event
            for event in recent_views
            if isinstance(event, dict)
            and event.get("product_id") is not None
        ]

        # Most recent first.

        recent_views.sort(
            key=lambda x: timestamp_to_unix(x.get("ts")),
            reverse=True,
        )

        recent_views = recent_views[:self.max_history]

        history_ids = [
            str(event["product_id"])
            for event in recent_views
        ]

        history_timestamps = [
            timestamp_to_unix(event.get("ts"))
            for event in recent_views
        ]

        history = self.encode_products(history_ids)

        history_length = len(history_ids)

        # -------------------------
        # 3. Padding
        # -------------------------

        def pad(tensor, length, value=0):

            padding = length - len(tensor)

            if padding <= 0:
                return tensor

            return torch.cat([
                tensor,
                torch.full(
                    (padding,),
                    value,
                    dtype=tensor.dtype,
                ),
            ])

        history = {
            key: pad(value, self.max_history)
            for key, value in history.items()
        }

        history_timestamps = pad(
            torch.tensor(
                history_timestamps,
                dtype=torch.float32,
            ),
            self.max_history,
        )

        history_mask = torch.zeros(
            self.max_history,
            dtype=torch.bool,
        )

        history_mask[:history_length] = True

        # -------------------------
        # 4. Candidates
        # -------------------------

        shown = row["shown"]

        candidates = self.encode_products(shown)

        positive_ids = set(row["positives"])

        positive_mask = torch.tensor(
            [pid in positive_ids for pid in shown],
            dtype=torch.bool,
        )

        # -------------------------
        # 5. Return
        # -------------------------

        return {

            "request_id": row["request_id"],

            "device": torch.tensor(device, dtype=torch.long),
            "region": torch.tensor(region, dtype=torch.long),
            "page": torch.tensor(page, dtype=torch.long),

            "request_timestamp": torch.tensor(
                timestamp_to_unix(row["ts"]),
                dtype=torch.float32,
            ),

            "history_category": history["category"],
            "history_subcategory": history["subcategory"],
            "history_brand": history["brand"],
            "history_price": history["price"],

            "history_timestamps": history_timestamps,
            "history_mask": history_mask,

            "candidate_category": candidates["category"],
            "candidate_subcategory": candidates["subcategory"],
            "candidate_brand": candidates["brand"],
            "candidate_price": candidates["price"],

            "positive_mask": positive_mask,

            "candidate_ids": shown,
        }

In [118]:
def collate_impressions(batch):

    if not batch:
        raise ValueError("Cannot collate an empty batch.")

    batch_size = len(batch)

    max_candidates = max(
        len(x["candidate_category"])
        for x in batch
    )

    candidate_fields = [
        "candidate_category",
        "candidate_subcategory",
        "candidate_brand",
        "candidate_price",
    ]

    fixed_fields = [
        "device",
        "region",
        "page",
        "request_timestamp",
        "history_category",
        "history_subcategory",
        "history_brand",
        "history_price",
        "history_timestamps",
        "history_mask",
    ]

    result = {}

    for field in fixed_fields:
        result[field] = torch.stack([
            x[field] for x in batch
        ])

    for field in candidate_fields:

        dtype = batch[0][field].dtype

        padded = torch.zeros(
            batch_size,
            max_candidates,
            dtype=dtype,
        )

        for i, item in enumerate(batch):

            n = len(item[field])

            padded[i, :n] = item[field]

        result[field] = padded

    candidate_mask = torch.zeros(
        batch_size,
        max_candidates,
        dtype=torch.bool,
    )

    positive_mask = torch.zeros_like(candidate_mask)

    for i, item in enumerate(batch):

        n = len(item["candidate_category"])

        candidate_mask[i, :n] = True

        positive_mask[i, :n] = item["positive_mask"]

    result["candidate_mask"] = candidate_mask
    result["positive_mask"] = positive_mask

    result["request_id"] = [
        x["request_id"] for x in batch
    ]

    result["candidate_ids"] = [
        x["candidate_ids"] for x in batch
    ]

    return result

In [119]:
def build_encoder(values):
    """
    0 is reserved for unknown/missing values.
    Real values start at 1.
    """

    unique_values = sorted({
        str(v)
        for v in values
        if pd.notna(v) and str(v) != ""
    })

    return {
        value: idx + 1
        for idx, value in enumerate(unique_values)
    }


def encode_value(value, encoder):
    if value is None:
        return 0

    if pd.isna(value):
        return 0

    return encoder.get(str(value), 0)

In [120]:
catalog = sql("""
    SELECT
        product_id,
        brand,
        category,
        subcategory,
        price_eur,
        is_active
    FROM shop.catalog
""")

# Build categorical encoders.

category_encoder = build_encoder(
    catalog["category"]
)

subcategory_encoder = build_encoder(
    catalog["subcategory"]
)

brand_encoder = build_encoder(
    catalog["brand"]
)


# Build product feature lookup.

product_features = {}

for row in catalog.itertuples():

    if pd.isna(row.product_id):
        continue

    product_id = str(row.product_id)

    price = pd.to_numeric(
        row.price_eur,
        errors="coerce",
    )

    if pd.isna(price):
        price = 0.0

    product_features[product_id] = {
        "category": encode_value(
            row.category,
            category_encoder,
        ),

        "subcategory": encode_value(
            row.subcategory,
            subcategory_encoder,
        ),

        "brand": encode_value(
            row.brand,
            brand_encoder,
        ),

        "price": float(max(price, 0.0)),
    }

In [121]:
def build_context_encoders(examples):

    devices = []
    regions = []
    pages = []

    for row in examples:
        context = row.get("context") or {}

        devices.append(context.get("device"))
        regions.append(context.get("region"))
        pages.append(context.get("page"))

    return {
        "device": build_encoder(devices),
        "region": build_encoder(regions),
        "page": build_encoder(pages),
    }

In [122]:
# -------------------------
# 1. Load examples
# -------------------------

examples = download_training_data()


# -------------------------
# 2. Build context encoders
# -------------------------

context_encoders = build_context_encoders(examples)


# -------------------------
# 3. Build dataset
# -------------------------

dataset = ImpressionDataset(
    examples=examples,
    product_features=product_features,
    context_encoders=context_encoders,
    max_history=MAX_HISTORY,
)


# -------------------------
# 4. Build DataLoader
# -------------------------

train_loader = DataLoader(
    dataset,
    batch_size=256,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_impressions,
    pin_memory=True,
)


# -------------------------
# 5. Construct model
# -------------------------

product_tower = ProductTower(
    num_categories=len(category_encoder) + 1,
    num_subcategories=len(subcategory_encoder) + 1,
    num_brands=len(brand_encoder) + 1,

    category_embedding_dim=16,
    subcategory_embedding_dim=32,
    brand_embedding_dim=32,

    hidden_dim=HIDDEN_1_DIM,
    output_dim=EMBEDDING_DIM,
)


user_tower = UserTower(
    num_devices=len(context_encoders["device"]) + 1,
    num_regions=len(context_encoders["region"]) + 1,
    num_pages=len(context_encoders["page"]) + 1,

    product_embedding_dim=EMBEDDING_DIM,

    device_embedding_dim=4,
    region_embedding_dim=8,
    page_embedding_dim=4,

    hidden_dim=256,
    output_dim=EMBEDDING_DIM,

    recency_tau=24.0,
)


model = TwoTowers(
    product_tower=product_tower,
    user_tower=user_tower,
)

In [124]:
def multi_positive_loss(
    logits,
    positive_mask,
    candidate_mask,
):

    # Every valid impression must have at least
    # one candidate and one positive.

    valid_rows = (
        candidate_mask.any(dim=1)
        & positive_mask.any(dim=1)
    )

    if not valid_rows.any():
        raise ValueError("Batch contains no valid impressions.")

    logits = logits[valid_rows]
    candidate_mask = candidate_mask[valid_rows]
    positive_mask = positive_mask[valid_rows]

    logits = logits.masked_fill(
        ~candidate_mask,
        float("-inf"),
    )

    positive_logits = logits.masked_fill(
        ~positive_mask,
        float("-inf"),
    )

    log_all = torch.logsumexp(logits, dim=1)

    log_positive = torch.logsumexp(
        positive_logits,
        dim=1,
    )

    return (log_all - log_positive).mean()

In [125]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

In [123]:
def train(
    model,
    train_loader,
    optimizer,
    num_epochs=30, # 10
    device="cuda",
):

    model.to(device)

    history = {
        "train_loss": [],
    }

    for epoch in range(num_epochs):

        model.train()

        total_loss = 0.0
        num_batches = 0

        for batch in train_loader:

            def move(key):
                return batch[key].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            # -------------------------
            # 1. User context
            # -------------------------

            device_features = move("device")
            region = move("region")
            page = move("page")

            # -------------------------
            # 2. Historical products
            # -------------------------

            history_category = move("history_category")
            history_subcategory = move("history_subcategory")
            history_brand = move("history_brand")
            history_price = move("history_price")

            history_timestamps = move("history_timestamps")
            history_mask = move("history_mask")

            request_timestamp = move("request_timestamp")

            # -------------------------
            # 3. Candidates
            # -------------------------

            candidate_category = move("candidate_category")
            candidate_subcategory = move("candidate_subcategory")
            candidate_brand = move("candidate_brand")
            candidate_price = move("candidate_price")

            candidate_mask = move("candidate_mask")
            positive_mask = move("positive_mask")

            # -------------------------
            # 4. Forward
            # -------------------------

            logits = model(
                device_features,
                region,
                page,

                history_category,
                history_subcategory,
                history_brand,
                history_price,
                history_timestamps,
                history_mask,

                request_timestamp,

                candidate_category,
                candidate_subcategory,
                candidate_brand,
                candidate_price,
            )

            # -------------------------
            # 5. Loss
            # -------------------------

            loss = multi_positive_loss(
                logits,
                positive_mask,
                candidate_mask,
            )

            # -------------------------
            # 6. Backpropagation
            # -------------------------

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

        epoch_loss = total_loss / max(num_batches, 1)

        history["train_loss"].append(epoch_loss)

        print(
            f"Epoch [{epoch + 1}/{num_epochs}] "
            f"- Loss: {epoch_loss:.6f}"
        )

    return model, history

In [127]:
model, history = train(
    model=model,
    train_loader=train_loader,
    optimizer=optimizer,
    num_epochs=100,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

Epoch [1/100] - Loss: 1.974380
Epoch [2/100] - Loss: 1.947414
Epoch [3/100] - Loss: 1.913716
Epoch [4/100] - Loss: 1.880056
Epoch [5/100] - Loss: 1.843175
Epoch [6/100] - Loss: 1.801198
Epoch [7/100] - Loss: 1.750219
Epoch [8/100] - Loss: 1.697845
Epoch [9/100] - Loss: 1.645054
Epoch [10/100] - Loss: 1.590595
Epoch [11/100] - Loss: 1.517195
Epoch [12/100] - Loss: 1.466546
Epoch [13/100] - Loss: 1.408530
Epoch [14/100] - Loss: 1.346938
Epoch [15/100] - Loss: 1.302481
Epoch [16/100] - Loss: 1.257946
Epoch [17/100] - Loss: 1.190810
Epoch [18/100] - Loss: 1.129695
Epoch [19/100] - Loss: 1.083339
Epoch [20/100] - Loss: 1.045721
Epoch [21/100] - Loss: 1.026206
Epoch [22/100] - Loss: 0.958642
Epoch [23/100] - Loss: 0.940612
Epoch [24/100] - Loss: 0.909272
Epoch [25/100] - Loss: 0.849296
Epoch [26/100] - Loss: 0.836862
Epoch [27/100] - Loss: 0.797970
Epoch [28/100] - Loss: 0.784992
Epoch [29/100] - Loss: 0.746541
Epoch [30/100] - Loss: 0.722199
Epoch [31/100] - Loss: 0.692139
Epoch [32/100] - 

##### Comparaciones de embeddings

In [128]:
def compare_products(model, product_id_a, product_id_b):

    device = next(model.parameters()).device

    product_a = product_features[str(product_id_a)]
    product_b = product_features[str(product_id_b)]

    model.eval()

    with torch.no_grad():

        def get_embedding(product):
            return model.product_tower(
                category=torch.tensor(
                    [product["category"]], device=device
                ),
                subcategory=torch.tensor(
                    [product["subcategory"]], device=device
                ),
                brand=torch.tensor(
                    [product["brand"]], device=device
                ),
                price=torch.tensor(
                    [product["price"]], dtype=torch.float32, device=device
                ),
            )

        embedding_a = get_embedding(product_a)
        embedding_b = get_embedding(product_b)

        similarity = F.cosine_similarity(
            embedding_a,
            embedding_b,
        ).item()

    print(f"Pair: A: {product_id_a}, B: {product_id_b}. Similarity: {similarity:.4f}")

    return similarity

In [129]:
compare_products(
    model,
    product_id_a="p_001897",
    product_id_b="p_001d7f",
)

Pair: A: p_001897, B: p_001d7f. Similarity: -0.0099


-0.009875250980257988

In [130]:
sql("SELECT product_id, brand, category, subcategory, price_eur FROM shop.catalog LIMIT 20")

,product_id,brand,category,subcategory,price_eur
0,p_001897,Vedo,Sports & Outdoors,Yoga Mats,14.99
1,p_00196b,Zore,Fashion,Jackets,4.99
2,p_001a20,Sanine,Electronics,Laptop Sleeves,57.99
3,p_001d7f,Vebexa,Sports & Outdoors,Running Shoes,29.99
4,p_0032c5,Nibe,Home & Kitchen,Cutting Boards,26.99
5,p_003840,Nika,Beauty & Health,Vitamins,46.99
6,p_004caa,Zofiri,Fashion,Socks,11.99
7,p_00675e,Rixa,Pets,Dog Toys,226.99
8,p_007935,Tixa,Toys & Baby,Strollers,34.99
9,p_008500,Luzo,Home & Kitchen,Coffee Machines,1.99


In [131]:
for a, b in [
    ("p_001897", "p_001d7f"),   # Yoga Mats vs Running Shoes
    ("p_00bfcf", "p_00c919"),   # Sneakers
    ("p_007935", "p_0147f5"),   # Strollers
    ("p_001a20", "p_013a18"),     # Laptop sleeves vs baby bottles
]:
    compare_products(model, a, b)

Pair: A: p_001897, B: p_001d7f. Similarity: -0.0099
Pair: A: p_00bfcf, B: p_00c919. Similarity: 0.3661
Pair: A: p_007935, B: p_0147f5. Similarity: 0.2979
Pair: A: p_001a20, B: p_013a18. Similarity: 0.0692


___

##### Testeos

In [132]:
device = next(model.parameters()).device

model.eval()

# All products with available features.
all_product_ids = list(product_features.keys())

with torch.inference_mode():

    product_embeddings = []

    for start in range(0, len(all_product_ids), 4096):

        ids = all_product_ids[start:start + 4096]
        features = [product_features[p] for p in ids]

        embeddings = model.product_tower(
            category=torch.tensor(
                [p["category"] for p in features],
                dtype=torch.long,
                device=device,
            ),
            subcategory=torch.tensor(
                [p["subcategory"] for p in features],
                dtype=torch.long,
                device=device,
            ),
            brand=torch.tensor(
                [p["brand"] for p in features],
                dtype=torch.long,
                device=device,
            ),
            price=torch.tensor(
                [p["price"] for p in features],
                dtype=torch.float32,
                device=device,
            ),
        )

        product_embeddings.append(embeddings)

    product_embeddings = torch.cat(product_embeddings)

# Product ID -> embedding index
product_index = {
    pid: i for i, pid in enumerate(all_product_ids)
}

# Active catalog candidates
active_ids = [
    str(pid)
    for pid in catalog.loc[
        catalog["is_active"] == True, "product_id"
    ]
    if str(pid) in product_index
]

active_indices = torch.tensor(
    [product_index[pid] for pid in active_ids],
    dtype=torch.long,
    device=device,
)

candidate_embeddings = product_embeddings[active_indices]

print("Products embedded:", len(all_product_ids))
print("Active candidates:", len(active_ids))

Products embedded: 5000
Active candidates: 5000


In [133]:
def two_tower(row):

    context = row.context
    views = row.recent_views

    # ---------------------------------
    # 1. Encode session context
    # ---------------------------------

    device_id = encode_value(
        context["device"], context_encoders["device"]
    )

    region_id = encode_value(
        context["region"], context_encoders["region"]
    )

    page_id = encode_value(
        context["page"], context_encoders["page"]
    )

    # ---------------------------------
    # 2. Encode interaction history
    # ---------------------------------

    history_ids = []
    history_timestamps = []

    for view in views:

        pid = str(view["product_id"])

        if pid not in product_index:
            continue

        history_ids.append(pid)

        history_timestamps.append(
            pd.Timestamp(view["ts"]).value / 1e9
        )

    request_timestamp = (
        pd.Timestamp(row.ts).value / 1e9
    )

    # Handle empty history.
    if not history_ids:

        history_indices = [0]
        history_timestamps = [request_timestamp]
        history_mask = [False]

    else:

        history_indices = [
            product_index[pid] for pid in history_ids
        ]

        history_mask = [True] * len(history_indices)

    history_embeddings = product_embeddings[
        torch.tensor(
            history_indices,
            dtype=torch.long,
            device=device,
        )
    ].unsqueeze(0)

    # ---------------------------------
    # 3. Encode user
    # ---------------------------------

    with torch.inference_mode():

        user_embedding = model.user_tower(
            device=torch.tensor(
                [device_id], dtype=torch.long, device=device
            ),

            region=torch.tensor(
                [region_id], dtype=torch.long, device=device
            ),

            page=torch.tensor(
                [page_id], dtype=torch.long, device=device
            ),

            recent_product_embeddings=history_embeddings,

            recent_timestamps=torch.tensor(
                [history_timestamps],
                dtype=torch.float32,
                device=device,
            ),

            history_mask=torch.tensor(
                [history_mask],
                dtype=torch.bool,
                device=device,
            ),

            request_timestamp=torch.tensor(
                [request_timestamp],
                dtype=torch.float32,
                device=device,
            ),
        )

        # ---------------------------------
        # 4. Rank active catalog
        # ---------------------------------

        scores = (
            candidate_embeddings @ user_embedding.squeeze(0)
        )

        # Exclude previously viewed products.
        viewed = set(history_ids)

        if viewed:
            viewed_indices = [
                i for i, pid in enumerate(active_ids)
                if pid in viewed
            ]

            if viewed_indices:
                scores[viewed_indices] = -float("inf")

        top_indices = torch.topk(
            scores,
            k=min(10, len(active_ids)),
        ).indices.cpu().tolist()

    return [
        active_ids[i] for i in top_indices
    ]

In [134]:
from time import time

In [135]:
POLICIES = {"baseline": baseline, "two_tower": two_tower}

def evaluate(policy, df):
    print("Evaluating", policy)
    t0 = time()
    recs = [policy(r) for r in df.itertuples()]
    clicked = df.clicked.to_numpy()
    has_click = df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has_click].mean()
    first_shown = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first_shown)])
    reward = (clicked == first_shown).astype(float)            # click on the slot-1 product
    w = match / df.propensity.str[0].to_numpy()
    result = {"hit_rate@10": hit, "IPS_ctr_slot1": (w * reward).mean(),
            "SNIPS_ctr_slot1": (w * reward).sum() / max(w.sum(), 1e-9), "slot1_matches": int(match.sum())}
    print(f"Took {time() - t0} seconds.")
    return result

old_slot1 = (test.clicked == test.shown.str[0]).mean()
print(f"old system, observed slot-1 CTR: {old_slot1:.4f}")
pd.DataFrame({name: evaluate(p, test) for name, p in POLICIES.items()}).T

old system, observed slot-1 CTR: 0.0109
Evaluating <function baseline at 0x731ddb464180>
Took 8.775416374206543 seconds.
Evaluating <function two_tower at 0x731e94263560>
Took 62.434855222702026 seconds.


,hit_rate@10,IPS_ctr_slot1,SNIPS_ctr_slot1,slot1_matches
baseline,0.390290,0.006030,0.041677,3887.0
two_tower,0.060754,0.001957,0.022862,784.0
